In [ ]:
import numpy as np
from qutip import *
from scipy.linalg import sqrtm, eigvalsh
from scipy.stats import linregress
from numba import njit, prange
import pickle
import os
import gc

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display, Math

## Fidelity
Generic definition : 
$$ \mathcal{F}\left( \rho, \sigma \right) = \left( Tr \left[ \sqrt{ \sqrt{\rho} \sigma \sqrt{\rho} }\right] \right)^{2} $$ 
Definition for $ \rho $ Pure State and $ \sigma $ Mixed State : 
$$ \mathcal{F}\left( \rho, \sigma \right) = \langle \psi_{\rho} | \sigma | \psi_{\rho} \rangle $$
Definition for Pure State : 
$$ \mathcal{F}\left( \rho, \sigma \right) = |\langle \psi_{\rho} | \psi_{\sigma} \rangle|^{2} $$
Definition for Qubits : 
$$ \mathcal{F}\left( \rho, \sigma \right) = Tr\left( \rho \, \sigma \right) + 2 \sqrt{Det\left ( \rho \right) \, Det\left ( \sigma \right)} $$
## Trace Distance
Generic definition : 
$$ \mathcal{T}\left( \rho, \sigma \right) = \frac{1}{2} Tr \left[ \sqrt{\left( \rho - \sigma \right)^{\dagger} \left( \rho - \sigma  \right)} \right] $$
### Relationship : Fuchs-van de Graaf inequality
$$ 1 - \sqrt{\mathcal{F}\left( \rho, \sigma \right)} \leq \mathcal{T}\left( \rho, \sigma \right) \leq \sqrt{1 - \mathcal{F}\left( \rho, \sigma \right)} $$

In [ ]:
def fidelity_generic(rho, sigma):
    """
    Calculate the quantum fidelity between two generic density matrices.
    Formula: F(rho, sigma) = ( Tr[ sqrt( sqrt(rho) * sigma * sqrt(rho) ) ] )^2
    
    This version avoids scipy.linalg.sqrtm to prevent RuntimeWarnings, 
    using stable eigenvalue decomposition instead.
    
    Parameters:
        rho (numpy.ndarray): First density matrix (NxN).
        sigma (numpy.ndarray): Second density matrix (NxN).
        
    Returns:
        float: The fidelity between rho and sigma (real number between 0 and 1).
    """
    # 1. Square root of rho using eigenvalue decomposition
    evals_rho, evecs_rho = np.linalg.eigh(rho)
    # Truncate any negative noise to 0.0 before taking the square root
    evals_rho = np.maximum(evals_rho, 0.0) 
    sqrt_rho = evecs_rho @ np.diag(np.sqrt(evals_rho)) @ evecs_rho.conj().T
    
    # 2. Inner matrix: sqrt(rho) * sigma * sqrt(rho)
    inner_matrix = sqrt_rho @ sigma @ sqrt_rho
    
    # Force exact Hermiticity to remove any small imaginary noise
    inner_matrix = 0.5 * (inner_matrix + inner_matrix.conj().T)
    
    # 3. Trace of the square root is the sum of the square roots of the eigenvalues
    evals_inner = eigvalsh(inner_matrix)
    # Again, truncate negative noise to 0.0 before square root
    evals_inner = np.maximum(evals_inner, 0.0)
    
    fidelity = np.sum(np.sqrt(evals_inner))**2
    
    # Ensure numerical errors do not push fidelity slightly above 1.0
    return min(1.0, fidelity)
    

In [ ]:
@njit
def fidelity_qubit(rho, sigma):
    """
    Calculate the exact quantum fidelity between two single-qubit (2x2) density matrices.
    Formula: F(rho, sigma) = Tr(rho * sigma) + 2 * sqrt(Det(rho) * Det(sigma))
    """
    # Trace of the matrix product
    tr_term = np.real(np.trace(rho @ sigma))
    
    # Determinants of the two density matrices
    det_rho = np.real(np.linalg.det(rho))
    det_sigma = np.real(np.linalg.det(sigma))
    
    # FIX NUMERICO: Tronchiamo a 0 gli eventuali valori negativi infinitesimi
    det_rho = max(0.0, det_rho)
    det_sigma = max(0.0, det_sigma)
    
    # Calculate fidelity using the analytical formula for qubits
    fidelity = tr_term + 2.0 * np.sqrt(det_rho * det_sigma)
    
    return fidelity

In [ ]:
@njit
def fidelity_qubit_single_term(p00, p11, c10, c01, sigma):
    """
    Build the density matrix from its elements and then
    Calculate the exact quantum fidelity between two single-qubit (2x2) density matrices.
    Formula: F(rho, sigma) = Tr(rho * sigma) + 2 * sqrt(Det(rho) * Det(sigma))
    """
    # 1. Density Matrix build up 
    rho = np.zeros((2, 2), dtype=np.complex128)
    rho[0,0] = p00
    rho[0,1] = c01
    rho[1,0] = c10
    rho[1,1] = p11

    # 2. Trace of the matrix product (Calculated explicitly for 2x2 to avoid Numba issues)
    prod = rho @ sigma
    tr_term = prod[0,0].real + prod[1,1].real
    
    # 3. Determinants of the two 2x2 density matrices (ad - bc)
    det_rho = (rho[0,0] * rho[1,1] - rho[0,1] * rho[1,0]).real
    det_sigma = (sigma[0,0] * sigma[1,1] - sigma[0,1] * sigma[1,0]).real
    
    # 4. Numerical FIX: Truncate to 0 any infinitesimal negative values
    det_rho = max(0.0, det_rho)
    det_sigma = max(0.0, det_sigma)
    
    # 5. Calculate fidelity using the analytical formula for qubits
    fidelity = tr_term + 2.0 * np.sqrt(det_rho * det_sigma)
    
    return fidelity

In [ ]:

def trace_distance_generic(rho, sigma):
    """
    Calculate the Trace Distance between two generic density matrices.
    Formula: T(rho, sigma) = 1/2 * Tr[ sqrt( (rho - sigma)^dagger * (rho - sigma) ) ]
    
    Parameters:
        rho (numpy.ndarray): First density matrix (NxN).
        sigma (numpy.ndarray): Second density matrix (NxN).
        
    Returns:
        float: The trace distance between rho and sigma (real number between 0 and 1).
    """
    # Difference between the matrices
    diff = rho - sigma
    
    # Force exact Hermiticity to remove numerical noise
    diff = 0.5 * (diff + diff.conj().T)
    
    # Calculate the eigenvalues of the strictly Hermitian matrix 'diff'
    eigenvalues = eigvalsh(diff)
    
    # Trace distance is half the sum of the absolute eigenvalues
    t_dist = 0.5 * np.sum(np.abs(eigenvalues))
    
    # Ensure it stays within physical bounds
    return min(1.0, t_dist)
    

In [ ]:
@njit
def trace_distance_qubit(rho, sigma):
    """
    Calculate the exact Trace Distance between two single-qubit (2x2) density matrices.
    For a 2x2 traceless Hermitian matrix (rho - sigma), Det(diff) = -lambda^2 <= 0.
    Therefore, the Trace Distance simplifies to sqrt(-Det(rho - sigma)).
    
    Parameters:
        rho (numpy.ndarray): First density matrix (2x2).
        sigma (numpy.ndarray): Second density matrix (2x2).
        
    Returns:
        float: The trace distance between rho and sigma.
    """
    # Difference between the matrices
    diff = rho - sigma
    
    # Determinant of the difference
    det_diff = np.real(np.linalg.det(diff))
    
    # Since det_diff should be <= 0, -det_diff should be >= 0.
    # We use max(0.0, ...) to truncate any negative noise before applying sqrt.
    val_under_sqrt = max(0.0, -det_diff)
    
    t_dist = np.sqrt(val_under_sqrt)
    
    return min(1.0, t_dist)
    

In [ ]:
@njit
def trace_distance_qubit_single_term(p00, p11, c10, c01, sigma):
    """
    Build the density matrix from its elements and then
    Calculate the exact Trace Distance between two single-qubit (2x2) density matrices.
    For a 2x2 traceless Hermitian matrix (rho - sigma), Det(diff) = -lambda^2 <= 0.
    Therefore, the Trace Distance simplifies to sqrt(-Det(rho - sigma)).
    
    Parameters:
        rho (numpy.ndarray): First density matrix (2x2).
        sigma (numpy.ndarray): Second density matrix (2x2).
        
    Returns:
        float: The trace distance between rho and sigma.
    """
    # Density Matrix build up 
    rho = np.zeros((2, 2), dtype=np.complex128)
    rho[0,0] = p00
    rho[0,1] = c01
    rho[1,0] = c10
    rho[1,1] = p11
    
    # Difference between the matrices
    diff = rho - sigma
    
    # Determinant of the difference
    det_diff = (diff[0,0]*diff[1,1] - diff[0,1]*diff[1,0]).real
    
    # Since det_diff should be <= 0, -det_diff should be >= 0.
    # We use max(0.0, ...) to truncate any negative noise before applying sqrt.
    val_under_sqrt = max(0.0, -det_diff)
    
    t_dist = np.sqrt(val_under_sqrt)
    
    return min(1.0, t_dist)

In [ ]:
@njit
def compute_metrics_over_time(pop_10, pop_01, coh_1001, coh_0110, rho_lindblad_array):
    """
    Computes fidelity and trace distance over all time steps using Numba in C-speed.
    """
    time_steps = len(pop_10)
    
    fidelity_arr = np.zeros(time_steps)
    trace_dist_arr = np.zeros(time_steps)
    
    for t in range(time_steps):
        fidelity_arr[t] = fidelity_qubit_single_term(
            pop_10[t], pop_01[t], coh_1001[t], coh_0110[t], rho_lindblad_array[t]
        )
        trace_dist_arr[t] = trace_distance_qubit_single_term(
            pop_10[t], pop_01[t], coh_1001[t], coh_0110[t], rho_lindblad_array[t]
        )
        
    return fidelity_arr, trace_dist_arr

## Results Analysis

In [ ]:
# ====================================
# Physical & Simulation Parameters
# ====================================

# Time parameters
dt = 0.01
tf = 100.0
time_steps = int(tf / dt)

# List of Number of trajectories to analyze
N_traj_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 20000]        

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION :    
#   'QJ' → Quantum Jump
#   'SD' → State Diffusion
# ============================================================


MODE = 'QJ'   # Switch this to 'SD' 

# Configuration mapping based on MODE
_cfg = {
    'QJ': {
        'Input_file': "../Results/Data/Complete_rho/pop_1.0_result_mode_QJ_dt0p010000_Ntraj20000.npz",
        'Output_dir': "../Results/Plot/Fidelity/Avg/QJ" 
    },
    'SD': {
        'Input_file': "../Results/Data/Complete_rho/pop_1.0_result_mode_SD_dt0p010000_Ntraj20000.npz",
        'Output_dir': "../Results/Plot/Fidelity/Avg/SD"
    },
}

# Apply the selected configuration
cfg = _cfg[MODE]

# Set the global Input and Output_dir dynamically based on the current mode
Input_file = cfg['Input_file']
Output_dir = cfg['Output_dir']

# Create the output directory if it doesn't exist
os.makedirs(Output_dir, exist_ok=True)

print(f"--- Configuration Setup ---")
print(f"Current mode : {MODE}")
print(f"Input Data   : {cfg['Input_file']}")
print(f"Output Plots : {Output_dir}")

In [ ]:
# ===========================
# General Setup for Plotting
# ===========================

# Global Style Settings (Matplotlib rcParams)
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': ':',
    'figure.autolayout': True # plt.tight_layout()
})

# Automatic Figure Saving Function
def save_fig(fig, filename):
    """
    Saves the figure in both PNG or PDF
    """
    path_png = os.path.join(Output_dir, f"{filename}.png")
    # path_pdf = os.path.join(Output_dir, f"{filename}.pdf")  # save in pdf
    
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    # fig.savefig(path_pdf, bbox_inches='tight') # save in pdf
    print(f"Figure saved in: {Output_dir}/{filename}")

### Data Extraction

In [ ]:
# ------------
# Load Results
# ------------
data = np.load(Input_file)

times = data['times']
time_steps = len(times)

# ------------
# Extract data
# ------------

# Extract Lindblad populations
rho_lindblad = data['rho_list_lindblad']
# lindblad_00 = np.real(rho_lindblad[:, 0, 0])
# lindblad_11 = np.real(rho_lindblad[:, 1, 1])
# lindblad_01 = rho_lindblad[:, 0, 1]

# Extract Trajectories data 
pop_00 = data['pop_00']
pop_11 = data['pop_11']
coh_01 = data['coh_01']
coh_10 = np.conj(coh_01)

print("--- Data Loading Completed ---")

# ==============================
# Density Matrix reconstruction
# ==============================

N_traj_tot = pop_00.shape[1]

rho_complete = np.zeros((time_steps, N_traj_tot, 2, 2), dtype=np.complex128)
rho_complete[:, :, 0, 0] = pop_00
rho_complete[:, :, 1, 1] = pop_11
rho_complete[:, :, 0, 1] = coh_01
rho_complete[:, :, 1, 0] = np.conj(coh_01)


In [ ]:
# ===================================================================
# Fidelity and Trace Distance Analysis over time for different N_traj
# ===================================================================

# Initialize simple lists 
fidelity_results_list = []
trace_dist_results_list = []

for N in N_traj_list:
    print(f"Processing N = {N} trajectories...")
    
    # Calculate the mean over the first N trajectories
    pop_00_m   = np.mean(pop_00[:, :N], axis=1)
    pop_11_m   = np.mean(pop_11[:, :N], axis=1)
    coh_01_m = np.mean(coh_01[:, :N], axis=1)
    coh_10_m = np.mean(coh_10[:, :N], axis=1)

    # Compute metrics using your Numba functions
    fid_results, td_results = compute_metrics_over_time(
        pop_00_m, pop_11_m, coh_10_m, coh_01_m, rho_lindblad
    )
    
    # Append the resulting 1D arrays to the lists
    fidelity_results_list.append(fid_results)
    trace_dist_results_list.append(td_results)

print("Computation fully processed and memory cleared.")

## Fidelity and Trace Distance calculation

In [ ]:
# ===================================================
# MAX & MEAN INFIDELITY & TRACE DISTANCE CALCULATION 
# ===================================================

print("Starting Infidelity and Trace Distance metrics calculation...")

# Lists to store the calculated scalar metrics for plotting
max_infidelity_results = []
mean_infidelity_results = []
max_trace_distance_results = []
mean_trace_distance_results = []

# Loop through the indices of the lists
for i in range(len(N_traj_list)):
    # Extract the averaged time-evolution arrays for this specific N_traj
    fid_time_array = fidelity_results_list[i]
    td_time_array = trace_dist_results_list[i]
        
    # Calculate infidelity: 1 - Fidelity
    infidelity_array = 1.0 - fid_time_array
        
    # Calculate the maximum and temporal mean of the metrics
    max_inf = np.max(infidelity_array)
    mean_inf = np.mean(infidelity_array)
    
    max_td = np.max(td_time_array)
    mean_td = np.mean(td_time_array)
        
    # Append the computed scalar values to the lists
    max_infidelity_results.append(max_inf)
    mean_infidelity_results.append(mean_inf)
    max_trace_distance_results.append(max_td)
    mean_trace_distance_results.append(mean_td)

# Convert lists to NumPy arrays for easier plotting (Highly recommended!)
max_infidelity_results = np.array(max_infidelity_results)
mean_infidelity_results = np.array(mean_infidelity_results)
max_trace_distance_results = np.array(max_trace_distance_results)
mean_trace_distance_results = np.array(mean_trace_distance_results)

print("Infidelity and Trace Distance max/mean values computed successfully.")

## Plot

In [ ]:
%matplotlib ipympl
from IPython.display import Image, display
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, IntSlider
import matplotlib.ticker as ticker

In [ ]:
plt.close('all')
plt.plot(fidelity_results_list[0])
plt.show()

In [ ]:
plt.close('all')
plt.plot(trace_dist_results_list[0])
plt.show()

### Fidelity average over time for different Number of Trajectories

In [ ]:
# ==========================
# MEAN INFIDELITY VS N_TRAJ
# ==========================

plt.close('all')

plt.figure()

y_values_mean = mean_infidelity_results
    
# Plot the curve for this specific angle
plt.plot(N_traj_list, y_values_mean, marker='o', linestyle='-', label=f'Mean Infidelity ({MODE}')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log(1 - \langle \mathcal{F} \rangle_t)$")
plt.title(r"Convergence of $(1 - \langle \mathcal{F} \rangle_t)$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Avg_Infidelity_in_time_{MODE}"
save_fig(plt.gcf(), filename)


plt.show()

### Maximal InFidelity for different Number of Trajectories

In [ ]:
# =========================
# MAX INFIDELITY VS N_TRAJ
# =========================

plt.figure()
y_values_max = max_infidelity_results
    
# Plot the curve for this specific angle
plt.plot(N_traj_list, y_values_max, marker='s', linestyle='-', label=f'Max Infidelity ({MODE})')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log((1 - \mathcal{F})_{\max})$")
plt.title(r"Convergence of $(1 - \mathcal{F})_{\max}$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Max_Infidelity_{MODE}"
save_fig(plt.gcf(), filename)

plt.show()

### Trace Distance average over time for different Number of Trajectories

In [ ]:
# ==============================
# MEAN TRACE DISTANCE VS N_TRAJ
# ==============================

plt.close('all')

plt.figure()

y_values_mean = mean_trace_distance_results
    
# Plot the curve for this specific angle
plt.plot(N_traj_list, y_values_mean, marker='o', linestyle='-', label=f'Mean Trace Distance ({MODE})')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log(\langle T \rangle_t)$")
plt.title(r"Convergence of $\langle T \rangle_t$ vs Trajectories")

# Logarithmic scale for both axes
# plt.xscale('log')
# plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Avg_Trace_Distance_in_time_{MODE}"
save_fig(plt.gcf(), filename)


plt.show()

### Maximal Trace Distance for different Number of Trajectories

In [ ]:
# =============================
# MAX TRACE DISTANCE VS N_TRAJ
# =============================

plt.figure()

y_values_max = max_trace_distance_results
    
# Plot the curve for this specific angle
plt.plot(N_traj_list, y_values_max, marker='s', linestyle='-', label=f'Max Trace Distance ({MODE})')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log(T_{\max})$")
plt.title(r"Convergence of $T_{\max}$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Max_Trace_Distance_{MODE}"
save_fig(plt.gcf(), filename)

plt.show()

## Data Fitting

### Fidelity Avg in time

In [ ]:
from scipy.stats import linregress

# ===========================================
# FIT & PLOT: MEAN INFIDELITY VS TRAJECTORIES
# ===========================================

# 1. Gather all valid data points (excluding exact zeros for log10)
valid_N = []
valid_metric = []

# Iterate directly over the lists/arrays generated in the previous step
for N, metric_val in zip(N_traj_list, mean_infidelity_results):
    if metric_val > 0:  # Data must be strictly positive to compute log10
        valid_N.append(N)
        valid_metric.append(metric_val)

# 2. Convert lists to numpy arrays and apply log10 scale
log_N = np.log10(valid_N)
log_metric = np.log10(valid_metric)

# 3. Perform Linear Regression on the log-log data
slope, intercept, r_value, p_value, std_err = linregress(log_N, log_metric)

print("--- Fit Results: Mean Infidelity ---")
print(f"Slope (m):     {slope:.4f}")  
print(f"Intercept (q): {intercept:.4f}")
print(f"R-squared:     {r_value**2:.4f}\n")

# 4. Plotting
plt.close('all')
fig = plt.figure(figsize=(8, 6))

# Scatter plot of the valid simulation data points
plt.scatter(log_N, log_metric, color='blue', alpha=0.7, s=50, label='Simulation Data')

# Theoretical fit line construction
x_fit = np.linspace(min(log_N), max(log_N), 100)
y_fit = slope * x_fit + intercept

# Plot the linear regression line
plt.plot(x_fit, y_fit, color='red', linewidth=2, 
         label=f'Linear Fit: $y = {slope:.2f}x {intercept:+.2f}$')

# Formatting the plot with LaTeX labels
plt.xlabel(r"$\log_{10}(N_{\text{traj}})$")
plt.ylabel(r"$\log_{10}(\langle 1 - \mathcal{F} \rangle_t)$")
plt.title(f"Scaling Fit: Mean Infidelity vs Trajectories ({MODE})")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# Save the figure dynamically (uncomment when ready)
filename = f"Fit_Mean_Infidelity_{MODE}"
save_fig(fig, filename)

plt.show()

### Trace Distance FittinAvg in time

In [ ]:
# ================================================
# FIT & PLOT: MEAN TRACE DISTANCE VS TRAJECTORIES
# ================================================

# 1. Gather all valid data points (excluding exact zeros for log10)
valid_N = []
valid_metric = []

# Iterate directly over the lists/arrays generated in the previous step
for N, metric_val in zip(N_traj_list, mean_trace_distance_results):
    if metric_val > 0:  # Data must be strictly positive to compute log10
        valid_N.append(N)
        valid_metric.append(metric_val)

# 2. Convert lists to numpy arrays and apply log10 scale
log_N = np.log10(valid_N)
log_metric = np.log10(valid_metric)

# 3. Perform Linear Regression on the log-log data
slope, intercept, r_value, p_value, std_err = linregress(log_N, log_metric)

print("--- Fit Results: Mean Trace Distance ---")
print(f"Slope (m):     {slope:.4f}")  
print(f"Intercept (q): {intercept:.4f}")
print(f"R-squared:     {r_value**2:.4f}\n")

# 4. Plotting
plt.close('all')
fig = plt.figure(figsize=(8, 6))

# Scatter plot of the valid simulation data points
plt.scatter(log_N, log_metric, color='blue', alpha=0.7, s=50, label='Simulation Data')

# Theoretical fit line construction
x_fit = np.linspace(min(log_N), max(log_N), 100)
y_fit = slope * x_fit + intercept

# Plot the linear regression line
plt.plot(x_fit, y_fit, color='red', linewidth=2, 
         label=f'Linear Fit: $y = {slope:.2f}x {intercept:+.2f}$')

# Formatting the plot with LaTeX labels
plt.xlabel(r"$\log_{10}(N_{\text{traj}})$")
plt.ylabel(r"$\log_{10}(\langle T \rangle_t)$")
plt.title(f"Scaling Fit: Mean Trace Distance vs Trajectories ({MODE})")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# Save the figure dynamically (uncomment when ready)
filename = f"Fit_Mean_Trace_Distance_{MODE}"
save_fig(fig, filename)

plt.show()